In [1]:
"""The main class that supplies other files"""
import kagglehub
import pickle
import os
import joblib
import pandas as pd
import numpy as np
import spacy
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import f1_score
from sklearn.preprocessing import FunctionTransformer

In [3]:
import kagglehub
cwd=os.getcwd()
os.chdir(cwd)
# Download latest version
path = kagglehub.dataset_download("jackksoncsie/spam-email-dataset")
print(f'path to download, {path}')

In [4]:
!pip install -U spacy --quiet
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 23.1 MB/s  0:00:00.9 MB/s eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
df= pd.read_csv(f'{path}/emails.csv')

In [5]:
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [8]:
df.spam=df.spam.map({0:'Spam', 1:'Not Spam'})
print(df.head())
df.text.str.len().sum()

                                                text      spam
0  Subject: naturally irresistible your corporate...  Not Spam
1  Subject: the stock trading gunslinger  fanny i...  Not Spam
2  Subject: unbelievable new homes made easy  im ...  Not Spam
3  Subject: 4 color printing special  request add...  Not Spam
4  Subject: do not have money , get software cds ...  Not Spam


np.int64(8917171)

In [22]:
nlp=spacy.load('en_core_web_sm') # Function to clean and process the data for training
def extract(df):
    #df=pd.DataFrame(x,columns=['text'])
    def lemma(text): 
        doc=nlp(''.join(text)) 
        tok=[token.lemma_ for token in doc 
             if not token.is_punct and not token.is_stop] 
        return ' '.join(tok) 
    df['text']=df['text'].str.lower().apply(lemma)
    return df

In [34]:
df_one= extract(df.copy())

In [36]:
print(df_one['text'].str.len().sum())
print(df.spam.value_counts(normalize=True))  #Checking to see how unbalanced is the dataset
df.head()

5807365
spam
0    0.761173
1    0.238827
Name: proportion, dtype: float64


,text,spam
0,subject naturally irresistible corporate ident...,1
1,subject stock trading gunslinger fanny merri...,1
2,subject unbelievable new home easy m want ...,1
3,subject 4 color print special request additi...,1
4,subject money software cd software compatibi...,1


In [37]:
Xtrain, Xtest, ytrain, ytest=train_test_split(df_one['text'].values,df_one['spam'].values,
                                              test_size=.1,
                                              random_state=42,
                                              shuffle=True)

#text_proc = FunctionTransformer(extract)
pipe=Pipeline([('transformer',TfidfVectorizer()),
               ('model', LinearSVC())
              ]
             )
params = [
    {
        'model': [LinearSVC()],
        'model__C': np.linspace(0.5, 2, 6),
        'model__class_weight': [{0: 1, 1: v} for v in range(1, 6)]
    },
    {
        'model': [LogisticRegression(max_iter=1000)],
        'model__C': np.linspace(0.5, 2, 6),
        'model__class_weight': [{0: 1, 1: v} for v in range(1, 6)]
    }
    ]
grid= GridSearchCV(pipe,
                 param_grid=params,
                 cv=5,
                 scoring=make_scorer(f1_score),
                 refit=True,
                verbose=10
                 )

In [38]:
print(Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape)

(5155,) (573,) (5155,) (573,)


In [88]:
model_1= grid.fit(Xtrain, ytrain)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
[CV 1/5; 1/60] START model=LinearSVC(), model__C=0.5, model__class_weight={0: 1, 1: 1}
[CV 1/5; 1/60] END model=LinearSVC(), model__C=0.5, model__class_weight={0: 1, 1: 1};, score=0.992 total time=   0.3s
[CV 2/5; 1/60] START model=LinearSVC(), model__C=0.5, model__class_weight={0: 1, 1: 1}
[CV 2/5; 1/60] END model=LinearSVC(), model__C=0.5, model__class_weight={0: 1, 1: 1};, score=0.989 total time=   0.2s
[CV 3/5; 1/60] START model=LinearSVC(), model__C=0.5, model__class_weight={0: 1, 1: 1}
[CV 3/5; 1/60] END model=LinearSVC(), model__C=0.5, model__class_weight={0: 1, 1: 1};, score=0.997 total time=   0.2s
[CV 4/5; 1/60] START model=LinearSVC(), model__C=0.5, model__class_weight={0: 1, 1: 1}
[CV 4/5; 1/60] END model=LinearSVC(), model__C=0.5, model__class_weight={0: 1, 1: 1};, score=0.992 total time=   0.2s
[CV 5/5; 1/60] START model=LinearSVC(), model__C=0.5, model__class_weight={0: 1, 1: 1}
[CV 5/5; 1/60] END model=Linear

In [90]:
model_1.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('transformer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [92]:
f1_score(model_1.best_estimator_.predict(Xtest), ytest)

0.9898305084745763

In [ ]:
t= [r"Subject: naturally irresistible your corporate identity  lt is really hard to recollect a company : the"]
doc= nlp(''.join(t))
for token in doc:
    print(token.lemma_)

In [94]:
mod.best_estimator_.predict([r"""Hello Micah,We have received your application for the Data Analytics Specialist position. 
Thank you for your interest. We'll be in touch once we've been able to review your application.
In the meantime, why not learn more about us
At BT we believe in empowering our people to thrive and make a real impact."""])

array([1])

### Model depolyment

In [51]:
with open('trained_model', 'wb') as model:
    pickle.dump(model_1, model)


def prediction(text):
    with open('trained_model', 'rb') as model:
        model=pickle.load(model)
        print('Spam') if model.best_estimator_.predict(text)==1 else print('Not Spam') 

In [62]:
prediction([r"""Hello Micah,We have received your application for the Data Analytics Specialist position. 
Thank you for your interest. We'll be in touch once we've been able to review your application.
In the meantime, why not learn more about us
At BT we believe in empowering our people to thrive and make a real impact."""])

Spam


In [14]:
Xtrain2, Xtest2, ytrain2, ytest2=train_test_split(df['text'].values,df['spam'].values,
                                              test_size=.1,
                                              random_state=42,
                                              shuffle=True)

#text_proc = FunctionTransformer(extract)
pipe=Pipeline([('transformer',TfidfVectorizer()),
               ('model', LinearSVC())
              ]
             )
params = [
    {
        'model': [LinearSVC()],
        'model__C': np.linspace(0.5, 2, 6),
        'model__class_weight': [{'Spam': 1, 'Not Spam': v} for v in range(1, 6)]
    },
    {
        'model': [LogisticRegression(max_iter=1000)],
        'model__C': np.linspace(0.5, 2, 6),
        'model__class_weight': [{'Spam': 1, 'Not Spam': v} for v in range(1, 6)]
    }
    ]


In [21]:
import joblib
import os

pipe2 = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=30000,
        sublinear_tf=True
    )),
    ("model", LinearSVC())
])

model_2= GridSearchCV(pipe2,
                 param_grid=params,
                 cv=5,
                scoring=f1_score,
                refit=True
                 )

In [22]:
model_2.fit(Xtrain2, ytrain2)

/Users/imohekpenyong/anaconda3/envs/end2end/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/imohekpenyong/anaconda3/envs/end2end/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/Users/imohekpenyong/anaconda3/envs/end2end/lib/python3.14/site-packages/sklearn/utils/_param_validation.py", line 196, in wrapper
    params = func_sig.bind(*args, **kwargs)
  File "/Users/imohekpenyong/anaconda3/envs/end2end/lib/python3.14/inspect.py", line 3236, in bind
    return self._bind(args, kwargs)
           ~~~~~~~~~~^^^^^^^^^^^^^^
  File "/Users/imohekpenyong/anaconda3/envs/end2end/lib/python3.14/inspect.py", line 3160, in _bind
    raise TypeError(
        'too many positional arguments') from Non

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...LinearSVC())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model': [LinearSVC()], 'model__C': array([0.5, 0....4, 1.7, 2. ]), 'model__class_weight': [{'Not Spam': 1, 'Spam': 1}, {'Not Spam': 2, 'Spam': 1}, ...]}, {'model': [LogisticRegre...max_iter=1000)], 'model__C': array([0.5, 0....4, 1.7, 2. ]), 'model__class_weight': [{'Not Spam': 1, 'Spam': 1}, {'Not Spam': 2, 'Spam': 1}, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",<function f1_...t 0x16591a1f0>
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that 

In [23]:
model_2.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [33]:
model_2.score

<bound method BaseSearchCV.score of GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tfidf',
                                        TfidfVectorizer(max_features=30000,
                                                        min_df=2,
                                                        ngram_range=(1, 2),
                                                        sublinear_tf=True)),
                                       ('model', LinearSVC())]),
             param_grid=[{'model': [LinearSVC()],
                          'model__C': array([0.5, 0.8, 1.1, 1.4, 1.7, 2. ]),
                          'model__class_weight': [{'Not Spam': 1, 'Spam': 1},
                                                  {'Not Spam': 2, 'Spam': 1},
                                                  {'Not Spam': 3, 'Spam': 1},
                                                  {'Not Spam': 4, 'Spam': 1},
                                                  {'Not Spam': 5, 'Spam': 1}]},
                 

In [31]:
model_2.best_estimator_.score( Xtest2, ytest2)

0.9947643979057592

In [39]:
string=r"""Hello Micah,
We have received your application for the Data Analytics Specialist position. 
Thank you for your interest. We'll be in touch once we've been able to review your application.
In the meantime, why not learn more about us
At BT we believe in empowering our people to thrive and make a real impact."""

In [40]:
model_2.best_estimator_.predict([string])

array(['Spam'], dtype=object)

In [30]:
path = os.path.join(os.path.dirname(os.getcwd()), "models")
joblib.dump(model_2.best_estimator_, f'{path}/spam_model.joblib')

['/Users/imohekpenyong/sentiment_analysis/models/spam_model.joblib']